### After correctly defining users, story order, user journeys and cleaning the ANDROID events to the point which they are probably robust, lets start answering some questions

In [1]:
import gc
import pandas as pd
from pathlib import Path

In [2]:
BASE_DIR = Path.cwd().parent
events_path = BASE_DIR / "data/clean/Android_events.parquet"

In [3]:
ANDROID_events = pd.read_parquet(events_path)

In [4]:
story_path = BASE_DIR / "data/clean/story_order.parquet"

In [5]:
Story_order = pd.read_parquet(story_path)

In [16]:
Story_order.head(10)

,tour_id,tour_item_id,story_id,tour_item_title,story_title,in_events,story_position
0,51,2933,9635,Entrance,Directions,True,1
1,51,2933,9636,Entrance,A place to gather?,True,2
2,51,2933,9637,Entrance,The golden age,True,3
3,51,2933,9638,Entrance,The rise and fall,True,4
4,51,2933,9639,Entrance,Back to life,True,5
5,51,2933,49289,Entrance,Your journey begins,True,6
6,51,2934,9640,The Panathenaic Way,Directions,True,7
7,51,2934,9641,The Panathenaic Way,It’s my way or the Panathenaic Way,True,8
8,51,2935,9642,The Altar of the Twelve Gods,Directions,True,9
9,51,2935,9643,The Altar of the Twelve Gods,The navel of Athens,True,10


In [6]:
tour_path = BASE_DIR / "data/clean/tour_info.csv"

In [7]:
tour_info = pd.read_csv(tour_path)

In [8]:
ANDROID_events.head(10)

,event_datetime,event_name,platform,language,user_id,user_pseudo_id,tour_id,story_id,lang_id,audio_time_played,audio_time_paused
0,2025-07-01 06:28:39.799003,click_listen_now,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,107,<NA>,5,NaN,NaN
1,2025-07-01 06:28:51.839001,start_tour,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,107,<NA>,5,NaN,NaN
2,2025-07-01 06:28:52.212003,story_start,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,107,50606,5,NaN,NaN
3,2025-07-01 06:28:52.214004,collapse_player,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,107,50606,5,NaN,NaN
4,2025-07-01 06:29:08.945005,story_listened_20,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,107,50606,5,NaN,NaN
5,2025-07-01 06:29:24.110000,story_listened_40,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,107,50606,5,NaN,NaN
6,2025-07-01 06:29:24.110001,story_listened_60,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,107,50606,5,NaN,NaN
7,2025-07-01 06:29:26.903002,story_listened_80,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,107,50606,5,NaN,NaN
8,2025-07-01 06:29:35.503003,story_completed,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,107,50606,5,NaN,NaN
15,2025-07-01 06:30:07.737010,story_start,ANDROID,de-de,<NA>,c7f26129dd892cb7292f6283113726ad,107,42950,5,NaN,NaN


In [8]:
# ensure event_date exists
ANDROID_events["event_date"] = ANDROID_events["event_datetime"].dt.date

# create journey id
ANDROID_events["journey_id"] = (
    ANDROID_events
    .groupby(["user_pseudo_id", "tour_id", "event_date"])
    .ngroup()
)

In [9]:
ANDROID_events[["journey_id","user_pseudo_id","tour_id","event_date"]].drop_duplicates().head(20)

,journey_id,user_pseudo_id,tour_id,event_date
0,12146,c7f26129dd892cb7292f6283113726ad,107,2025-07-01
279,12147,c7f26129dd892cb7292f6283113726ad,226,2025-07-01
281,840,0e1e29466bf2de31e00fe7a989673c99,822,2025-06-30
290,841,0e1e29466bf2de31e00fe7a989673c99,822,2025-07-01
762,7295,767fa15a9558b669f47d8e67e719918c,284,2025-07-01
1354,10605,ada97a097eabb072201591dd267767cd,447,2025-06-30
1373,10456,aa95a709d2859cb0b976835344d7df10,278,2025-07-02
1376,10457,aa95a709d2859cb0b976835344d7df10,859,2025-07-04
1550,8879,8f1732d4bd0131b46860e311bdc1ff71,240,2025-06-30
2516,10841,b22d1794fe798eea9481aaaee845d122,240,2025-06-30


### How many journeys per user?

In [13]:
journeys_per_user = (
    ANDROID_events
    .groupby("user_pseudo_id")["journey_id"]
    .nunique()
    .reset_index(name="num_journeys")
)

journeys_per_user.head()

,user_pseudo_id,num_journeys
0,0000968c3b3d3f7db948beb9a8eab054,1
1,0000cbea45a05c83a469e0d5b9cc1bf3,1
2,0009031260d41ed70d7458f8324b0ad9,1
3,000b52af1c5a69eb96f0ef158a18dbe0,3
4,0033fc96d51050266ae2df2dcf553d63,1


In [29]:
journeys_per_user.num_journeys.sum()

15733

In [17]:
ANDROID_events.loc[
    (ANDROID_events["event_name"] == "story_completed") &
    (ANDROID_events["story_id"].isna())
]

,event_datetime,event_name,platform,language,user_id,user_pseudo_id,tour_id,story_id,lang_id,audio_time_played,audio_time_paused,event_date,journey_id
156009,2025-07-05 07:04:05.259027,story_completed,ANDROID,en-us,<NA>,2b48dd062081a9bc8a0fe0957e99a1ea,278,<NA>,8,NaN,NaN,2025-07-05,2742
3892061,2025-10-22 17:37:58.335026,story_completed,ANDROID,el-gr,<NA>,daef455039f060ae1ef5f0a23d550d33,621,<NA>,2,NaN,NaN,2025-10-22,13305
4260751,2025-10-10 19:52:11.554108,story_completed,ANDROID,en-gb,<NA>,0420f9ede0c62572833d921f316a8e2d,535,<NA>,2,NaN,NaN,2025-10-10,252
4862933,2025-10-24 06:50:47.120026,story_completed,ANDROID,el-gr,<NA>,6cdf33aae5fdc0695785a56c4199194c,893,<NA>,2,NaN,NaN,2025-10-24,6733
4862946,2025-10-24 06:51:06.914042,story_completed,ANDROID,el-gr,<NA>,6cdf33aae5fdc0695785a56c4199194c,908,<NA>,2,NaN,NaN,2025-10-24,6737
4863238,2025-10-24 08:45:05.503014,story_completed,ANDROID,el-gr,<NA>,6cdf33aae5fdc0695785a56c4199194c,908,<NA>,2,NaN,NaN,2025-10-24,6737
4863302,2025-10-24 10:39:37.182022,story_completed,ANDROID,el-gr,<NA>,6cdf33aae5fdc0695785a56c4199194c,908,<NA>,2,NaN,NaN,2025-10-24,6737
4863340,2025-10-24 11:37:57.253020,story_completed,ANDROID,el-gr,<NA>,6cdf33aae5fdc0695785a56c4199194c,908,<NA>,2,NaN,NaN,2025-10-24,6737
4863603,2025-10-24 12:26:18.290048,story_completed,ANDROID,el-gr,<NA>,6cdf33aae5fdc0695785a56c4199194c,908,<NA>,2,NaN,NaN,2025-10-24,6737


In [10]:
ANDROID_events = ANDROID_events.loc[
    ~(
        ANDROID_events["story_id"].isna() &
        ANDROID_events["event_name"].isin([
            "story_start",
            "story_listened_20",
            "story_listened_40",
            "story_listened_60",
            "story_listened_80",
            "story_completed"
        ])
    )
]

## How deeply do users actually consume the tour content? 

In [26]:
completed_stories = (
    ANDROID_events.loc[ANDROID_events["event_name"] == "story_completed"]
    .groupby(["journey_id", "tour_id"])["story_id"]
    .nunique()
    .reset_index(name="stories_completed")
)

tour_story_counts = (
    Story_order
    .groupby("tour_id")["story_id"]
    .nunique()
    .reset_index(name="total_stories")
)

journey_depth = (
    completed_stories
    .merge(tour_story_counts, on="tour_id", how="left")
)

journey_depth["depth_ratio"] = (
    journey_depth["stories_completed"] /
    journey_depth["total_stories"]
)

In [28]:
journey_depth["depth_ratio"].describe()

count    9210.000000
mean        0.306541
std         0.261265
min         0.004545
25%         0.084507
50%         0.238938
75%         0.474840
max         1.000000
Name: depth_ratio, dtype: float64

In [30]:
user_depth_features = (
    ANDROID_events[["journey_id", "user_pseudo_id"]]
    .drop_duplicates()
    .merge(journey_depth, on="journey_id", how="inner")
    .groupby("user_pseudo_id")
    .agg(
        depth_mean=("depth_ratio", "mean"),
        depth_median=("depth_ratio", "median"),
        depth_max=("depth_ratio", "max"),
        depth_min=("depth_ratio", "min"),
        total_completed_stories=("stories_completed", "sum"),
        total_tours_with_completion=("journey_id", "nunique")
    )
    .reset_index()
)

In [31]:
user_depth_features.head(10)

,user_pseudo_id,depth_mean,depth_median,depth_max,depth_min,total_completed_stories,total_tours_with_completion
0,0000968c3b3d3f7db948beb9a8eab054,0.070423,0.070423,0.070423,0.070423,5,1
1,0009031260d41ed70d7458f8324b0ad9,0.400000,0.400000,0.400000,0.400000,34,1
2,000b52af1c5a69eb96f0ef158a18dbe0,0.415493,0.415493,0.774648,0.056338,59,2
3,0033fc96d51050266ae2df2dcf553d63,0.027523,0.027523,0.027523,0.027523,3,1
4,00397bee728ba3bc1daae584f3e73d10,0.137500,0.137500,0.137500,0.137500,11,1
5,0039bb847bebed6c62c000e057fc4b50,0.323944,0.323944,0.323944,0.323944,23,1
6,0055d2695d93e219d46989e924b71974,0.075000,0.075000,0.075000,0.075000,6,1
7,0061c00a146af168cca572b40538a2a0,0.282609,0.282609,0.282609,0.282609,26,1
8,0076ca9ffc64b175dae1cdad029124dd,0.056235,0.056235,0.089744,0.022727,12,2
9,00810b9e3c1156eddc1326d14cc6ca64,0.051948,0.051948,0.051948,0.051948,4,1


In [32]:
user_full_completion = (
    ANDROID_events[["journey_id", "user_pseudo_id"]]
    .drop_duplicates()
    .merge(journey_depth, on="journey_id", how="inner")
)

user_full_completion["full_completion"] = (
    user_full_completion["depth_ratio"] == 1
).astype(int)

full_completion_stats = (
    user_full_completion
    .groupby("user_pseudo_id")["full_completion"]
    .mean()
    .reset_index(name="share_full_completion")
)

In [33]:
user_depth_features = user_depth_features.merge(
    full_completion_stats,
    on="user_pseudo_id",
    how="left"
)

In [35]:
user_depth_features.describe()

,depth_mean,depth_median,depth_max,depth_min,total_completed_stories,total_tours_with_completion,share_full_completion
count,7490.000000,7490.000000,7490.000000,7490.000000,7495.000000,7495.000000,7495.000000
mean,0.303393,0.302221,0.334425,0.273951,32.440027,1.230420,0.002792
std,0.240520,0.242041,0.264828,0.244170,31.003045,0.584136,0.048034
min,0.004545,0.004545,0.004545,0.004545,1.000000,1.000000,0.000000
25%,0.100917,0.100000,0.105418,0.069767,10.000000,1.000000,0.000000
50%,0.253521,0.250000,0.278171,0.207792,25.000000,1.000000,0.000000
75%,0.454545,0.454545,0.512993,0.412500,45.000000,1.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,399.000000,9.000000,1.000000


### lets put story order here as well

In [36]:
completed = ANDROID_events.loc[
    ANDROID_events["event_name"] == "story_completed",
    ["journey_id", "tour_id", "story_id"]
]

In [37]:
completed = completed.merge(
    Story_order[["tour_id", "story_id", "story_position"]],
    on=["tour_id", "story_id"],
    how="left"
)

In [38]:
dropoff = (
    completed
    .groupby(["journey_id","tour_id"])["story_position"]
    .max()
    .reset_index(name="max_story_position")
)

In [39]:
dropoff = dropoff.merge(
    tour_story_counts,
    on="tour_id",
    how="left"
)

In [40]:
dropoff["dropoff_ratio"] = (
    dropoff["max_story_position"] /
    dropoff["total_stories"]
)

In [41]:
dropoff.head(10)

,journey_id,tour_id,max_story_position,total_stories,dropoff_ratio
0,0,535,10.0,71.0,0.140845
1,2,512,40.0,85.0,0.470588
2,4,535,5.0,71.0,0.070423
3,5,535,71.0,71.0,1.000000
4,6,644,17.0,109.0,0.155963
5,7,820,36.0,80.0,0.450000
6,8,535,40.0,71.0,0.563380
7,11,240,64.0,80.0,0.800000
8,12,858,54.0,92.0,0.586957
9,14,107,157.0,220.0,0.713636


In [42]:
dropoff.loc[dropoff["dropoff_ratio"].isna()]

,journey_id,tour_id,max_story_position,total_stories,dropoff_ratio
1176,2022,190,NaN,NaN,NaN
1541,2643,208,NaN,NaN,NaN
3345,5695,208,NaN,NaN,NaN
3346,5696,208,NaN,NaN,NaN
4090,7004,208,NaN,NaN,NaN
6083,10369,208,NaN,NaN,NaN
6352,10826,208,NaN,NaN,NaN
6353,10827,208,NaN,NaN,NaN
7585,12932,190,NaN,NaN,NaN
7586,12933,190,NaN,NaN,NaN


i dont have info about tours 190 and 208

### How long are the journeys?

In [43]:
journey_time = (
    ANDROID_events
    .groupby("journey_id")["event_datetime"]
    .agg(["min","max"])
    .reset_index()
)

journey_time["total_time_seconds"] = (
    journey_time["max"] - journey_time["min"]
).dt.total_seconds()

In [44]:
journey_info = (
    ANDROID_events[["journey_id","user_pseudo_id","tour_id"]]
    .drop_duplicates()
)

In [45]:
journey_table = (
    journey_info
    .merge(journey_depth[["journey_id","depth_ratio"]], on="journey_id", how="left")
    .merge(journey_time[["journey_id","total_time_seconds"]], on="journey_id", how="left")
)

In [46]:
journey_table = journey_table.loc[
    journey_table["depth_ratio"].notna()
]

In [47]:
journey_table["total_time_minutes"] = (
    journey_table["total_time_seconds"] // 60
).astype(int)

journey_table = journey_table.drop(columns="total_time_seconds")

In [51]:
tour_info.head(10)

,tour_id,tour_name,tour_length,total_stories
0,51,Ancient Agora: the birth of democracy,140.0,113.0
1,55,National Archaeological Museum: the notable Gr...,100.0,82.0
2,79,The curious Oltrarno,NaN,19.0
3,86,"Delphi and Iera Chora, the steps of Parnassus",180.0,13.0
4,91,Piraeus: Hidden urban stories,110.0,51.0
5,107,﻿Knossos: Daily Life in the Minoan Era,100.0,83.0
6,121,On the prowl: the cats of Athens,30.0,30.0
7,126,Find your (queer) god in Mykonos,100.0,20.0
8,139,Delphi: the Google of the ancient world-old,50.0,50.0
9,141,Mani: The gates of Hades,NaN,20.0


In [52]:
journey_table = journey_table.merge(
    tour_info[["tour_id", "tour_length"]],
    on="tour_id",
    how="left"
)

In [53]:
journey_table["length_difference"] = (
    journey_table["total_time_minutes"] - journey_table["tour_length"]
)

In [55]:
weird_guy = ANDROID_events[ANDROID_events['user_pseudo_id'] == "7cc861b11e0ba24b114730bcfafc80bd"]

### How long are the journeys? One big table

In [11]:
stories_touched = (
    ANDROID_events.loc[ANDROID_events["event_name"] == "story_start"]
    .groupby("journey_id")["story_id"]
    .nunique()
    .reset_index(name="distinct_stories_touched")
)

stories_completed = (
    ANDROID_events.loc[ANDROID_events["event_name"] == "story_completed"]
    .groupby("journey_id")["story_id"]
    .nunique()
    .reset_index(name="distinct_stories_completed")
)

times_completed = (
    ANDROID_events.loc[ANDROID_events["event_name"] == "story_completed"]
    .groupby("journey_id")["story_id"]
    .count()
    .reset_index(name="total_times_completed")
)

events_with_order = ANDROID_events.merge(
    Story_order[["tour_id","story_id","story_position"]],
    on=["tour_id","story_id"],
    how="left"
)

max_depth = (
    events_with_order.loc[events_with_order["event_name"] == "story_completed"]
    .groupby(["journey_id","tour_id"])["story_position"]
    .max()
    .reset_index(name="max_story_completed_position")
)

dropoff = (
    events_with_order.loc[events_with_order["event_name"] == "story_start"]
    .groupby(["journey_id","tour_id"])["story_position"]
    .max()
    .reset_index(name="dropoff_position")
)



In [12]:
journey_base = (
    ANDROID_events[["journey_id","user_pseudo_id","tour_id"]]
    .drop_duplicates()
)

journey_stats = (
    journey_base
    .merge(stories_touched, on="journey_id", how="left")
    .merge(stories_completed, on="journey_id", how="left")
    .merge(times_completed, on="journey_id", how="left")
    .merge(max_depth, on=["journey_id","tour_id"], how="left")
    .merge(dropoff, on=["journey_id","tour_id"], how="left")
    .merge(tour_info[["tour_id","total_stories"]], on="tour_id", how="left")
)

journey_stats["depth_ratio"] = (
    journey_stats["distinct_stories_completed"] /
    journey_stats["total_stories"]
)

journey_stats["max_depth"] = (
    journey_stats["max_story_completed_position"] /
    journey_stats["total_stories"]
)

journey_stats["dropoff"] = (
    journey_stats["dropoff_position"] /
    journey_stats["total_stories"]
)



In [13]:
journey_stats.shape

(15733, 12)

In [14]:
funny_journeys = journey_stats.loc[
    journey_stats["distinct_stories_touched"].isna()
]

In [15]:
journey_stats = journey_stats.loc[
    journey_stats["distinct_stories_touched"].notna()
]

In [16]:
journey_stats.shape

(12150, 12)

In [93]:
journey_stats.describe()

,journey_id,tour_id,distinct_stories_touched,distinct_stories_completed,total_times_completed,max_story_completed_position,dropoff_position,total_stories,depth_ratio,max_depth,dropoff
count,12150.000000,12150.0,12150.000000,9222.000000,9222.0,9210.000000,12130.000000,12130.000000,9210.000000,9210.000000,12130.000000
mean,7837.720329,517.082716,30.064033,26.364997,29.172197,66.368947,63.963891,92.434130,0.306541,0.689103,0.668183
std,4528.039814,272.978634,26.542235,22.147638,25.360292,49.978141,51.216498,41.172109,0.261265,0.340396,0.379423
min,0.000000,51.0,1.000000,1.000000,1.0,1.000000,1.000000,12.000000,0.004545,0.009174,0.006667
25%,3920.500000,278.0,4.000000,7.000000,8.0,31.000000,21.000000,73.000000,0.084507,0.404018,0.266901
50%,7815.000000,535.0,25.000000,21.000000,23.0,64.000000,65.000000,80.000000,0.238938,0.835616,0.877273
75%,11748.500000,822.0,48.000000,41.000000,45.0,80.000000,80.000000,105.000000,0.474840,0.995455,1.000000
max,15732.000000,910.0,118.000000,111.000000,235.0,220.000000,229.000000,283.000000,1.000000,1.000000,1.000000


In [94]:
weird_journey = ANDROID_events[ANDROID_events['journey_id'] == 3933]

### Journey story sequentiality: Do users follow the intended story order or jump around?

In [17]:
# 1) keep only story_start events and attach story order
story_start_seq = (
    ANDROID_events.loc[ANDROID_events["event_name"] == "story_start",
                       ["journey_id", "user_pseudo_id", "tour_id", "story_id", "event_datetime"]]
    .merge(
        Story_order[["tour_id", "story_id", "story_position"]],
        on=["tour_id", "story_id"],
        how="left"
    )
    .sort_values(["journey_id", "event_datetime"])
)

# 2) previous story position inside each journey
story_start_seq["prev_story_position"] = (
    story_start_seq.groupby("journey_id")["story_position"].shift(1)
)

# 3) transition difference
story_start_seq["position_diff"] = (
    story_start_seq["story_position"] - story_start_seq["prev_story_position"]
)

# 4) flag sequential transitions (+1 means exact intended next story)
story_start_seq["is_sequential"] = (story_start_seq["position_diff"] == 1).astype(int)

# 5) count transitions per journey
journey_transitions = (
    story_start_seq.loc[story_start_seq["prev_story_position"].notna()]
    .groupby("journey_id")
    .agg(
        total_transitions=("is_sequential", "size"),
        sequential_transitions=("is_sequential", "sum")
    )
    .reset_index()
)

# 6) sequential order metric
journey_transitions["sequential_ratio"] = (
    journey_transitions["sequential_transitions"] /
    journey_transitions["total_transitions"]
)

# 7) attach journey identity
journey_sequentiality = (
    ANDROID_events[["journey_id", "user_pseudo_id", "tour_id"]]
    .drop_duplicates()
    .merge(journey_transitions, on="journey_id", how="left")
)

# optional: inspect
journey_sequentiality.head()

,journey_id,user_pseudo_id,tour_id,total_transitions,sequential_transitions,sequential_ratio
0,12146,c7f26129dd892cb7292f6283113726ad,107,81.0,62.0,0.765432
1,12147,c7f26129dd892cb7292f6283113726ad,226,NaN,NaN,NaN
2,840,0e1e29466bf2de31e00fe7a989673c99,822,1.0,0.0,0.000000
3,841,0e1e29466bf2de31e00fe7a989673c99,822,120.0,60.0,0.500000
4,7295,767fa15a9558b669f47d8e67e719918c,284,110.0,14.0,0.127273


In [18]:
journey_sequentiality.shape

(15733, 6)

In [19]:
journey_sequentiality = journey_sequentiality.loc[
    journey_sequentiality["total_transitions"].notna()
]

In [20]:
journey_sequentiality.shape

(10482, 6)

In [21]:
journey_stats = journey_stats.loc[
    journey_stats["journey_id"].isin(journey_sequentiality["journey_id"])
]

In [22]:
journey_stats = journey_stats.merge(
    journey_sequentiality[[
        "journey_id",
        "total_transitions",
        "sequential_transitions",
        "sequential_ratio"
    ]],
    on="journey_id",
    how="left"
)

In [23]:
# story_ids completed inside each journey
completed_in_journey = (
    ANDROID_events.loc[ANDROID_events["event_name"] == "story_completed", ["journey_id", "story_id"]]
    .drop_duplicates()
    .assign(completed_in_journey=1)
)

# add completion flag to the story_start sequence table
story_start_seq = story_start_seq.merge(
    completed_in_journey,
    on=["journey_id", "story_id"],
    how="left"
)

story_start_seq["completed_in_journey"] = story_start_seq["completed_in_journey"].fillna(0).astype(int)

# completed sequential transition:
# current story is sequential (+1 from previous) AND current story got completed
story_start_seq["is_completed_sequential"] = (
    (story_start_seq["position_diff"] == 1) &
    (story_start_seq["completed_in_journey"] == 1)
).astype(int)

# aggregate per journey
journey_seq_completed = (
    story_start_seq.loc[story_start_seq["prev_story_position"].notna()]
    .groupby("journey_id")
    .agg(
        completed_sequential_transitions=("is_completed_sequential", "sum")
    )
    .reset_index()
)

# ratio
journey_seq_completed = journey_seq_completed.merge(
    journey_sequentiality[["journey_id", "total_transitions"]],
    on="journey_id",
    how="left"
)

journey_seq_completed["sequential_completed_ratio"] = (
    journey_seq_completed["completed_sequential_transitions"] /
    journey_seq_completed["total_transitions"]
)

# merge into journey_stats
journey_stats = journey_stats.merge(
    journey_seq_completed[[
        "journey_id",
        "completed_sequential_transitions",
        "sequential_completed_ratio"
    ]],
    on="journey_id",
    how="left"
)

In [104]:
journey_stats.describe()

,journey_id,tour_id,distinct_stories_touched,distinct_stories_completed,total_times_completed,max_story_completed_position,dropoff_position,total_stories,depth_ratio,max_depth,dropoff,total_transitions,sequential_transitions,sequential_ratio,completed_sequential_transitions,sequential_completed_ratio
count,10482.000000,10482.0,10482.000000,9183.000000,9183.0,9183.000000,10482.000000,10482.000000,9183.000000,9183.000000,10482.000000,10482.000000,10482.000000,10482.000000,10482.000000,10482.000000
mean,7850.387426,515.344782,34.664282,26.454536,29.269302,66.468692,70.713604,92.906888,0.307402,0.690119,0.744105,54.791643,25.446957,0.427393,17.953349,0.279198
std,4529.989259,273.057235,25.726093,22.145616,25.361179,49.971101,49.472522,41.585709,0.261165,0.339764,0.334153,47.849300,24.818596,0.252389,20.997017,0.243070
min,0.000000,51.0,1.000000,1.000000,1.0,1.000000,1.000000,12.000000,0.004545,0.009174,0.006667,1.000000,0.000000,0.000000,0.000000,0.000000
25%,3918.500000,252.0,11.000000,7.000000,8.0,32.000000,40.000000,73.000000,0.084507,0.408451,0.521127,14.000000,4.000000,0.250000,1.000000,0.043478
50%,7853.500000,535.0,32.000000,22.000000,23.0,64.000000,71.000000,80.000000,0.239437,0.836735,0.948052,46.000000,19.000000,0.431818,10.000000,0.250000
75%,11764.750000,822.0,53.000000,41.000000,45.0,80.000000,82.000000,105.000000,0.475000,0.995455,1.000000,83.000000,40.000000,0.600000,28.000000,0.450666
max,15732.000000,910.0,118.000000,111.000000,235.0,220.000000,229.000000,283.000000,1.000000,1.000000,1.000000,642.000000,299.000000,1.000000,247.000000,1.000000


In [24]:
# rebuild previous story_id as well
story_start_seq = story_start_seq.sort_values(["journey_id", "event_datetime"]).copy()

story_start_seq["prev_story_id"] = (
    story_start_seq.groupby("journey_id")["story_id"].shift(1)
)

story_start_seq["prev_story_position"] = (
    story_start_seq.groupby("journey_id")["story_position"].shift(1)
)

story_start_seq["position_diff"] = (
    story_start_seq["story_position"] - story_start_seq["prev_story_position"]
)

# keep only rows that are exact sequential transitions
seq_only = story_start_seq.loc[
    story_start_seq["position_diff"] == 1,
    ["journey_id", "prev_story_id", "story_id", "completed_in_journey"]
].copy()

# DISTINCT sequential transitions
distinct_seq = (
    seq_only
    .drop_duplicates(subset=["journey_id", "prev_story_id", "story_id"])
)

# count distinct sequential transitions per journey
distinct_seq_counts = (
    distinct_seq
    .groupby("journey_id")
    .size()
    .reset_index(name="distinct_sequential_transitions")
)

# count DISTINCT completed sequential transitions per journey
distinct_completed_seq_counts = (
    distinct_seq.loc[distinct_seq["completed_in_journey"] == 1]
    .groupby("journey_id")
    .size()
    .reset_index(name="distinct_completed_sequential_transitions")
)

# merge into journey_stats
journey_stats = journey_stats.drop(
    columns=[
        "distinct_sequential_transitions",
        "distinct_completed_sequential_transitions",
        "global_sequential_coverage",
        "global_completed_sequential_coverage"
    ],
    errors="ignore"
)

journey_stats = journey_stats.merge(
    distinct_seq_counts,
    on="journey_id",
    how="left"
)

journey_stats = journey_stats.merge(
    distinct_completed_seq_counts,
    on="journey_id",
    how="left"
)

journey_stats["distinct_sequential_transitions"] = (
    journey_stats["distinct_sequential_transitions"].fillna(0)
)

journey_stats["distinct_completed_sequential_transitions"] = (
    journey_stats["distinct_completed_sequential_transitions"].fillna(0)
)

# max possible distinct sequential transitions in a full tour
journey_stats["max_possible_transitions"] = journey_stats["total_stories"] - 1

# corrected global ratios
journey_stats["global_sequential_coverage"] = (
    journey_stats["distinct_sequential_transitions"] /
    journey_stats["max_possible_transitions"]
)

journey_stats["global_completed_sequential_coverage"] = (
    journey_stats["distinct_completed_sequential_transitions"] /
    journey_stats["max_possible_transitions"]
)

### Story jumps forward or backwards stats

In [26]:
# keep only journeys that exist in journey_stats
jump_base = story_start_seq.loc[
    story_start_seq["journey_id"].isin(journey_stats["journey_id"])
].copy()

# keep only actual transitions
jump_transitions = jump_base.loc[
    jump_base["prev_story_position"].notna()
].copy()

# define jump types exactly like the old code
jump_transitions["is_forward_jump"] = (jump_transitions["position_diff"] > 1).astype(int)
jump_transitions["is_backward"] = (jump_transitions["position_diff"] < 0).astype(int)

# aggregate per journey
journey_jump_stats = (
    jump_transitions
    .groupby("journey_id")
    .agg(
        pct_forward_jump=("is_forward_jump", "mean"),
        pct_backward=("is_backward", "mean"),
        avg_jump_distance=("position_diff", "mean")
    )
    .reset_index()
)

journey_jump_stats.head()

,journey_id,pct_forward_jump,pct_backward,avg_jump_distance
0,0,0.190476,0.178571,0.785714
1,2,0.025000,0.025000,0.850000
2,4,0.000000,0.200000,-0.200000
3,5,0.142857,0.268908,-0.050420
4,6,0.222222,0.222222,1.111111


In [27]:
# if you want to merge it
# journey_stats = journey_stats.merge(
#     journey_jump_stats,
#     on="journey_id",
#     how="left"
# )

## Are users actively listening or passively letting audio play?

In [28]:
# events considered interactive
interactive_events = [
    "pause",
    "play",
    "forward_10",
    "backward_10",
    "next_story",
    "previous_story"
]

# keep only journeys used in analysis
events_subset = ANDROID_events.loc[
    ANDROID_events["journey_id"].isin(journey_stats["journey_id"])
].copy()

# flags
events_subset["is_interactive"] = events_subset["event_name"].isin(interactive_events)

events_subset["is_pause_play"] = events_subset["event_name"].isin([
    "pause", "play"
])

events_subset["is_forward"] = events_subset["event_name"].isin([
    "forward_10", "next_story"
])

events_subset["is_backward"] = events_subset["event_name"].isin([
    "backward_10", "previous_story"
])

# compute journey-level stats
journey_interactivity = (
    events_subset
    .groupby("journey_id")
    .agg(
        total_events=("event_name", "count"),
        interactive_events=("is_interactive", "sum"),
        pause_play_events=("is_pause_play", "sum"),
        forward_events=("is_forward", "sum"),
        backward_events=("is_backward", "sum")
    )
    .reset_index()
)

# overall interactivity
journey_interactivity["interactivity_pct"] = (
    journey_interactivity["interactive_events"] /
    journey_interactivity["total_events"]
)

# breakdown within interactive events
journey_interactivity["pause_play_pct"] = (
    journey_interactivity["pause_play_events"] /
    journey_interactivity["interactive_events"]
)

journey_interactivity["forward_pct"] = (
    journey_interactivity["forward_events"] /
    journey_interactivity["interactive_events"]
)

journey_interactivity["backward_pct"] = (
    journey_interactivity["backward_events"] /
    journey_interactivity["interactive_events"]
)

journey_interactivity.head()

,journey_id,total_events,interactive_events,pause_play_events,forward_events,backward_events,interactivity_pct,pause_play_pct,forward_pct,backward_pct
0,0,270,69,56,5,8,0.255556,0.811594,0.072464,0.115942
1,2,238,2,2,0,0,0.008403,1.000000,0.000000,0.000000
2,4,38,1,1,0,0,0.026316,1.000000,0.000000,0.000000
3,5,632,19,19,0,0,0.030063,1.000000,0.000000,0.000000
4,6,75,12,11,1,0,0.160000,0.916667,0.083333,0.000000


In [29]:
journey_stats = journey_stats.merge(
    journey_interactivity[["journey_id", "interactivity_pct"]],
    on="journey_id",
    how="left"
)

In [30]:
journey_raw = journey_stats.copy()

In [31]:
journey_stats = journey_stats.fillna(0)

In [32]:
journey_stats.describe()

,journey_id,tour_id,distinct_stories_touched,distinct_stories_completed,total_times_completed,max_story_completed_position,dropoff_position,total_stories,depth_ratio,max_depth,...,sequential_transitions,sequential_ratio,completed_sequential_transitions,sequential_completed_ratio,distinct_sequential_transitions,distinct_completed_sequential_transitions,max_possible_transitions,global_sequential_coverage,global_completed_sequential_coverage,interactivity_pct
count,10482.000000,10482.0,10482.000000,10482.000000,10482.0,10482.000000,10482.000000,10482.000000,10482.000000,10482.000000,...,10482.000000,10482.000000,10482.000000,10482.000000,10482.000000,10482.000000,10482.000000,10482.000000,10482.000000,10482.000000
mean,7850.387426,515.344782,34.664282,23.176111,25.642053,58.231444,70.713604,92.906888,0.269306,0.604595,...,25.446957,0.427393,17.953349,0.279198,21.888857,15.324175,91.906888,0.253555,0.177518,0.081915
std,4529.989259,273.057235,25.726093,22.486300,25.622108,51.646251,49.472522,41.585709,0.264602,0.390954,...,24.818596,0.252389,20.997017,0.243070,20.515091,17.356465,41.585709,0.236917,0.201748,0.084031
min,0.000000,51.0,1.000000,0.000000,0.0,0.000000,1.000000,12.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,11.000000,0.000000,0.000000,0.000000
25%,3918.500000,252.0,11.000000,3.000000,3.0,15.000000,40.000000,73.000000,0.038961,0.184354,...,4.000000,0.250000,1.000000,0.043478,4.000000,1.000000,72.000000,0.049383,0.012658,0.023056
50%,7853.500000,535.0,32.000000,17.000000,18.0,56.000000,71.000000,80.000000,0.192661,0.750000,...,19.000000,0.431818,10.000000,0.250000,16.000000,9.000000,79.000000,0.200000,0.105769,0.060216
75%,11764.750000,822.0,53.000000,38.000000,41.0,80.000000,82.000000,105.000000,0.434783,0.987500,...,40.000000,0.600000,28.000000,0.450666,34.000000,24.000000,104.000000,0.381579,0.277778,0.113511
max,15732.000000,910.0,118.000000,111.000000,235.0,220.000000,229.000000,283.000000,1.000000,1.000000,...,299.000000,1.000000,247.000000,1.000000,108.000000,104.000000,282.000000,1.000000,1.000000,0.846906


In [33]:
journey_stats.to_csv(BASE_DIR / 'data/clean/journey_stats.csv')